In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [2]:
import logging
import pandas as pd
import aiohttp
import asyncio
from datetime import datetime, timedelta
import requests
import json
import time

from src_oop.core.scraper import HTTPClient
from src_oop.core.utils_general import load_api_tokens, load_sima_land_tokens

In [4]:
class SimaLandClient:
    def __init__(self, token=None):
        if token:
            self.token = token
        else:
            self.token = load_sima_land_tokens()['ВЕКТОР']
        self.headers = {
                "x-api-key": self.token
            }

    def _download_all_categories_to_json(self):
        url = "https://www.sima-land.ru/api/v3/category/"
        headers = {"x-api-key": self.token}
        
        all_categories = []
        page = 1
        per_page = 50 # Оптимально для API
        
        print("🚀 Начинаю полную выгрузку категорий. Это может занять пару минут...")
        
        while True:
            params = {
                "page": page,
                "per-page": per_page,
                "is_active": 1 # Берем только те, где есть товары
            }
            
            try:
                response = requests.get(url, headers=headers, params=params)
                response.raise_for_status()
                data = response.json()
                
                items = data.get('items', [])
                if not items:
                    break
                
                # Сохраняем только самое важное, чтобы файл не весил 100Мб
                for item in items:
                    all_categories.append({
                        "id": item['id'],
                        "name": item['name'],
                        "level": item['level'],
                        "path": item['path'] # Поможет понять вложенность
                    })
                
                print(f"✅ Загружено страниц: {page}", end="\r")
                
                # Проверка на последнюю страницу
                if page >= data.get('_meta', {}).get('pageCount', 0):
                    break
                    
                page += 1
                time.sleep(0.05) # Минимальная задержка
                
            except Exception as e:
                print(f"\n❌ Ошибка на странице {page}: {e}")
                break

        # Сохраняем в файл
        with open("sima_categories.json", "w", encoding="utf-8") as f:
            json.dump(all_categories, f, ensure_ascii=False, indent=4)
        
        print(f"\n\nГотово! Всего собрано категорий: {len(all_categories)}")
        print("Данные сохранены в файл: sima_categories.json")

    def get_sim_land_items(self, category_id):
        url = "https://www.sima-land.ru/api/v3/item/"
        all_items = []
        last_id = 0  
        
        params = {
            "category_id": category_id,
            "has_balance": 1,
            "is_disabled": 0,
            "is_deleted": 0,
            "has_photo": 1,
            "per-page": 50,
            "sort": "id"  
        }

        print(f"🚀 Начинаю выгрузку товаров для категории {category_id}...")

        while True:
            # Используем альтернативную пагинацию
            params["id-greater-than"] = last_id
            
            try:
                response = requests.get(url, headers=self.headers, params=params)
                response.raise_for_status()
                data = response.json()
                
                items = data.get('items', [])
                if not items:
                    break     

                for item in items:
                    clean_item = {
                        "sid": item.get('sid'),
                        "name": item.get('name'),
                        "balance": item.get('balance', 0),
                        "price": item.get('price', 0),
                        "price_max": item.get('price_max', 0),
                        "boxtype_id": item.get('boxtype_id'),
                        "box_depth": item.get('box_depth', 0),
                        "box_width": item.get('box_width', 0),
                        "box_height": item.get('box_height', 0),
                        "width": item.get('width', 0),
                        "height": item.get('height', 0),    
                        "per_package": item.get('per_package', 0),
                        "photo_url": item.get('photoUrl', ''),
                    }
                    all_items.append(clean_item)
                
                # Обновляем last_id значением ID последнего товара в текущей пачке
                last_id = items[-1]['id']

                print(f"✅ Загружено товаров: {len(all_items)} (последний ID: {last_id})", end="\r")
                
                # Если пришло меньше, чем мы просили (per-page), значит товары кончились
                if len(items) < params["per-page"]:
                    break
                    
                time.sleep(0.1)
                
            except Exception as e:
                print(f"\n❌ Ошибка: {e}")
                break

        print(f"\nВыгрузка завершена. Итого: {len(all_items)} шт.")
        return all_items

In [5]:
client = SimaLandClient()
items = client.get_sim_land_items(82958)

if items:
    print(f"Первый товар: {items[0]['name']} - {items[0]['price']} руб.")

🚀 Начинаю выгрузку товаров для категории 82958...
✅ Загружено товаров: 136 (последний ID: 8215184)
Выгрузка завершена. Итого: 136 шт.
Первый товар: Защита роликовая ONLYTOP, р. S, цвет розовый - 249 руб.


In [9]:
df = pd.DataFrame(items)
df_filter = df[df['name'].str.contains("самокат", case=False, na=False)]
df_filter.head()

,sid,name,balance,price,price_max,boxtype_id,box_depth,box_width,box_height,width,height,per_package,photo_url
16,1738611,"Самокат городской GRAFFITI Technology 200, складной, колёса PU 200 мм, ABEC 7, амортизатор задний, чёрный",0,5999.0,5999,576,85.0,15.0,32.0,13.0,100,1,https://goods-photos.static1-sima-land.com/items/2058573/0/700.jpg?v=1740736479
28,5358842,"Самокат 2 в 1 GRAFFITI Kids, с корзинкой, колёса PU 120/75 мм, цвет серый",0,1699.0,1999,576,56.0,15.0,27.0,0.0,0,1,https://goods-photos.static1-sima-land.com/items/5744751/0/700.jpg?v=1758777032
29,5358843,"Самокат 2 в 1 с корзинкой GRAFFITI Kids, колёса PU 120/75 мм, розовый",0,2250.0,2250,576,56.0,15.0,27.0,0.0,0,1,https://goods-photos.static1-sima-land.com/items/5744752/0/700.jpg?v=1740981727
41,6913028,"Лыжи для самокатов-снегокатов GRAFFITI, пара, цвет розовый",12,449.0,449,574,38.0,9.0,18.0,9.0,18,1,https://goods-photos.static1-sima-land.com/items/6354827/0/700.jpg?v=1756210068
43,7339246,"Самокат GRAFFITI, складной, колёса PU 145 мм, ABEC 7, амортизатор передний, алюминиевый",11,3999.0,3999,576,68.0,11.0,26.0,0.0,0,1,https://goods-photos.static1-sima-land.com/items/6638137/0/700.jpg?v=1740982523


In [29]:
sima = SimaLandClient()
token = sima.token

category_id = 57332

headers = {
    "x-api-key": token
}
url = "https://www.sima-land.ru/api/v3/item/"
params_item = {
    "category_id": category_id,
    "has_balance": 1,
    "is_disabled": 0,
    "is_deleted": 0,
    "has_photo": 1,
}


url_category = "https://www.sima-land.ru/api/v3/category/"
name = "самокаты"
params_cat = {
    "id": category_id,
    "expand-root": {
        "level": 1
    }
}
url_attr = "https://www.sima-land.ru/api/v3/attr/"

In [23]:
path = r"sima_categories.json"
path_to_save = 'sima_goods.xlsx'
df_to_excel = pd.read_json(path)
df_to_excel = df_to_excel.to_excel(path_to_save)
df_to_excel.head()

AttributeError: 'NoneType' object has no attribute 'head'

In [15]:

headers = {"x-api-key": token}

# 1. Ищем все подкатегории внутри "Спорт и туризм" (level=2)
# Или ищем категорию, где в названии есть "самокаты"
url_category = "https://www.sima-land.ru/api/v3/category/"
params_cat = {
    "level": 5,          # Ищем на уровень ниже
    "path": "16",        # Указываем, что родитель - Спорт (ID 16)
    "is_active": 1
}

res = requests.get(url_category, headers=headers, params=params_cat)
categories = res.json().get('items', [])

In [16]:
for cat in categories:
    if "самокат" in cat['name'].lower():
        print(f"Найдена категория: {cat['name']} (ID: {cat['id']})")
    # else:
    #     print(f"Категория не подходит: {cat['name']} (ID: {cat['id']})")

Найдена категория: Самокаты и аксессуары (ID: 2208)


In [58]:
names = []
for item in data['items']:
    names.append((item['name'], item['id']))
print(names)

[('Аксессуары для волос', 1), ('Бижутерия', 2), ('Бизнес-сувениры', 3), ('Брелоки и подвески', 4), ('Талисманы и фэншуй', 5), ('Интерьерные сувениры', 7), ('Канцтовары', 8), ('Магниты и значки', 10), ('Подарочные наборы', 11), ('Посуда', 12), ('Приколы', 14), ('Спорт и туризм', 16), ('Сувениры с символикой стран и городов', 18), ('Оформление и организация праздника', 19), ('Подарочная упаковка', 20), ('Фоторамки и фотоальбомы', 21), ('Хозтовары', 22), ('Ободки', 26), ('Невидимки, зажимы и шпильки', 33), ('Резинки', 35), ('Крабы, бананы, заколки', 43), ('Наборы аксессуаров для волос', 50), ('Наклейки', 77), ('На телефон', 78), ('Термопосуда', 80), ('Туризм', 81), ('Термопосуда и термосумки', 82), ('Профессиональные праздники', 84), ('Военный', 85), ('Платки', 87), ('Ручки', 91), ('Ручки-приколы', 92), ('Стержни', 94), ('Карандаши', 96), ('Маркеры', 97), ('Визитницы', 98), ('Зеркала', 99), ('Наборы и подставки', 119), ('Блокноты и записные книжки', 121), ('Ежедневники', 124), ('Папки, си

In [23]:
for item in data['items']:
    print(item['name'])

Форма для выпечки и упаковки конфет бумажная UPAK LAND «Сердце», 3.5×2 см, одноразовая, зелёная
Форма для выпечки и упаковки конфет бумажная UPAK LAND «Цветочки», 3.5×2 см, одноразовая, фиолетовая
Форма для выпечки и упаковки конфет бумажная UPAK LAND «Клетка», 3.5×2 см, одноразовая, розовая
Форма для выпечки и упаковки конфет бумажная UPAK LAND «Цветок», 3.5×2 см, одноразовая, синяя
Подкнопочник, d=4/11 мм, прозрачный
Форма для выпечки кексов бумажная UPAK LAND «Горох», 3.5×2 см, одноразовая, разноцветная
Форма для выпечки и упаковки конфет бумажная UPAK LAND «Калейдоскоп», 3.5×2 см, одноразовая, жёлтая
Форма для выпечки и упаковки конфет бумажная UPAK LAND «Поцелуи», 3.5×2 см, одноразовая, белая
Форма для выпечки и упаковки конфет бумажная UPAK LAND «Ромашки», 3.5×2 см, одноразовая, разноцветная
Форма для выпечки и упаковки конфет бумажная UPAK LAND «Вечеринка», 3.5×2 см, одноразовая, разноцветная
Форма для выпечки и упаковки конфет бумажная UPAK LAND «Калейдоскоп», 3.5×2 см, однораз

In [ ]:
4632511502,
4909557968,
4910124200,
4910283426,
4920579467,
4922131394,
4923046896,
4923300292,
4923496655,
4923508987,
4923582481,
4925108429,
4925934392